# Multiple Linear Regression

## Study Notes: Assumptions of Linear Regression

A fitted regression line can appear convincing while still giving a misleading account of the data. **Anscombe's quartet** demonstrates this clearly: four datasets can have nearly identical summary statistics and the same fitted regression equation while displaying very different patterns when plotted. The lesson is simple: calculate the model, but also visualize the data and examine its diagnostics.

Linear regression relies on several assumptions. Violations do not all have the same consequence: some mainly affect coefficient interpretation and statistical inference, while others damage predictions or indicate that the model form is inappropriate.

### 1. Linearity

The expected value of the dependent variable should be a linear function of the model parameters and predictors. In practice, the residuals should not show a systematic curved pattern.

- **Check:** Scatterplots, residuals-versus-fitted plots, and partial-residual plots.
- **Warning sign:** Curves or other systematic structure in the residuals.
- **Possible remedies:** Transform variables, add polynomial or interaction terms, or use an appropriate nonlinear model.

### 2. Homoscedasticity

The residuals should have approximately constant variance across the range of fitted values. A funnel or cone shape indicates **heteroscedasticity**, meaning that prediction errors become more or less variable as the fitted value changes.

- **Check:** A residuals-versus-fitted plot; formal tests such as Breusch-Pagan can supplement the visual check.
- **Why it matters:** Heteroscedasticity can make conventional standard errors, confidence intervals, and hypothesis tests unreliable.
- **Possible remedies:** Transform the target, use heteroscedasticity-robust standard errors, or fit a weighted model when justified.

### 3. Approximately normal residuals

The normality assumption applies to the model's **errors or residuals**, not necessarily to the predictors or the dependent variable themselves. It is most important for reliable confidence intervals and hypothesis tests, especially in small samples; ordinary least-squares coefficient estimation does not itself require perfectly normal residuals.

- **Check:** A residual histogram and a normal Q-Q plot.
- **Warning signs:** Strong skewness, heavy tails, or large departures from the Q-Q reference line.
- **Possible remedies:** Investigate data quality and outliers, transform variables, or use inference methods suited to the observed error distribution.

### 4. Independence of observations and errors

Observations—and therefore their errors—should not systematically depend on one another. **Autocorrelation** commonly occurs in time-series, spatial, clustered, or repeated-measures data. For example, today's stock price is related to earlier prices, so treating daily observations as independent would be questionable.

- **Check:** Plot residuals in observation or time order and inspect their autocorrelation; the Durbin-Watson statistic is one diagnostic for certain forms of serial correlation.
- **Warning sign:** Cycles, runs, clusters, or other sequential patterns in residuals.
- **Possible remedies:** Model the time, group, or spatial structure explicitly, or use a method designed for correlated observations.

### 5. No perfect or severe multicollinearity

Predictors should not be exact linear combinations of one another, and very strong correlations among predictors deserve attention. Severe multicollinearity makes individual coefficients unstable, inflates their standard errors, and makes their separate effects difficult to interpret. It does not automatically mean that the model cannot predict well.

- **Check:** A predictor correlation matrix and variance inflation factors (VIFs).
- **Possible remedies:** Remove or combine redundant predictors, collect more data, use domain knowledge to select variables, or consider regularization.

### Additional check: outliers, leverage, and influence

An unusual observation can pull the fitted line toward itself and materially change the coefficients. Distinguish between a large residual, unusual predictor values (**high leverage**), and an observation that substantially changes the fitted model (**high influence**).

- **Check:** Standardized or studentized residuals, leverage values, and Cook's distance.
- **Do not remove observations automatically.** First verify data quality and use subject-matter knowledge to decide whether the observation is erroneous, outside the target population, or a valid and important case. Compare results with and without influential observations and document the decision.

### Practical checklist

1. Plot the raw variables before fitting the model.
2. Fit the regression and inspect residuals versus fitted values.
3. Examine a residual Q-Q plot and the residual distribution.
4. Check residual independence in the relevant time, spatial, or group order.
5. Assess predictor redundancy with correlations and VIFs.
6. Investigate influential observations rather than deleting them mechanically.
7. Validate predictive performance on data that was not used for training.

> **Course note:** The exercises may treat these assumptions as satisfied so the focus remains on learning the algorithms. In real projects, the assumptions must be checked against the data and the intended use of the model.

## Importing the libraries

In [0]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

## Importing the dataset

In [0]:
dataset = pd.read_csv('50_Startups.csv')
X = dataset.iloc[:, :-1].values
y = dataset.iloc[:, -1].values

In [3]:
print(X)

[[165349.2 136897.8 471784.1 'New York']
 [162597.7 151377.59 443898.53 'California']
 [153441.51 101145.55 407934.54 'Florida']
 [144372.41 118671.85 383199.62 'New York']
 [142107.34 91391.77 366168.42 'Florida']
 [131876.9 99814.71 362861.36 'New York']
 [134615.46 147198.87 127716.82 'California']
 [130298.13 145530.06 323876.68 'Florida']
 [120542.52 148718.95 311613.29 'New York']
 [123334.88 108679.17 304981.62 'California']
 [101913.08 110594.11 229160.95 'Florida']
 [100671.96 91790.61 249744.55 'California']
 [93863.75 127320.38 249839.44 'Florida']
 [91992.39 135495.07 252664.93 'California']
 [119943.24 156547.42 256512.92 'Florida']
 [114523.61 122616.84 261776.23 'New York']
 [78013.11 121597.55 264346.06 'California']
 [94657.16 145077.58 282574.31 'New York']
 [91749.16 114175.79 294919.57 'Florida']
 [86419.7 153514.11 0.0 'New York']
 [76253.86 113867.3 298664.47 'California']
 [78389.47 153773.43 299737.29 'New York']
 [73994.56 122782.75 303319.26 'Florida']
 [67532

## Encoding categorical data

In [0]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
ct = ColumnTransformer(transformers=[('encoder', OneHotEncoder(), [3])], remainder='passthrough')
X = np.array(ct.fit_transform(X))

In [5]:
print(X)

[[0.0 0.0 1.0 165349.2 136897.8 471784.1]
 [1.0 0.0 0.0 162597.7 151377.59 443898.53]
 [0.0 1.0 0.0 153441.51 101145.55 407934.54]
 [0.0 0.0 1.0 144372.41 118671.85 383199.62]
 [0.0 1.0 0.0 142107.34 91391.77 366168.42]
 [0.0 0.0 1.0 131876.9 99814.71 362861.36]
 [1.0 0.0 0.0 134615.46 147198.87 127716.82]
 [0.0 1.0 0.0 130298.13 145530.06 323876.68]
 [0.0 0.0 1.0 120542.52 148718.95 311613.29]
 [1.0 0.0 0.0 123334.88 108679.17 304981.62]
 [0.0 1.0 0.0 101913.08 110594.11 229160.95]
 [1.0 0.0 0.0 100671.96 91790.61 249744.55]
 [0.0 1.0 0.0 93863.75 127320.38 249839.44]
 [1.0 0.0 0.0 91992.39 135495.07 252664.93]
 [0.0 1.0 0.0 119943.24 156547.42 256512.92]
 [0.0 0.0 1.0 114523.61 122616.84 261776.23]
 [1.0 0.0 0.0 78013.11 121597.55 264346.06]
 [0.0 0.0 1.0 94657.16 145077.58 282574.31]
 [0.0 1.0 0.0 91749.16 114175.79 294919.57]
 [0.0 0.0 1.0 86419.7 153514.11 0.0]
 [1.0 0.0 0.0 76253.86 113867.3 298664.47]
 [0.0 0.0 1.0 78389.47 153773.43 299737.29]
 [0.0 1.0 0.0 73994.56 122782.75 3

## Splitting the dataset into the Training set and Test set

In [0]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 0)

## Training the Multiple Linear Regression model on the Training set

In [7]:
from sklearn.linear_model import LinearRegression
regressor = LinearRegression()
regressor.fit(X_train, y_train)

LinearRegression(copy_X=True, fit_intercept=True, n_jobs=None, normalize=False)

## Predicting the Test set results

In [8]:
y_pred = regressor.predict(X_test)
np.set_printoptions(precision=2)
print(np.concatenate((y_pred.reshape(len(y_pred),1), y_test.reshape(len(y_test),1)),1))

[[103015.2  103282.38]
 [132582.28 144259.4 ]
 [132447.74 146121.95]
 [ 71976.1   77798.83]
 [178537.48 191050.39]
 [116161.24 105008.31]
 [ 67851.69  81229.06]
 [ 98791.73  97483.56]
 [113969.44 110352.25]
 [167921.07 166187.94]]
